# 1b — Network Preparation

Prepares the directed road network of Serbia for use in criticality analysis. Outputs a single parquet file of directed edges with topology, AADT, speed, and free-flow travel time attributes.

---

## Inputs

| File | Description |
|------|-------------|
| `input_files/DeoniceRSDP-Jul2025_corrected_topology.parquet` | Raw road network (geometry + attributes: `oznaka_deo`, `smer_gdf1`, `kategorija`, `duzina_deo`, `stanje`, etc.) |
| `input_files/PGDS_2024.shp` | Official 2024 AADT traffic count data by road segment |
| `input_files/ne_10m_admin_0_countries.shp` | Natural Earth country boundaries (used to exclude Kosovo) |

## Outputs

| File | Description |
|------|-------------|
| `intermediate_results/PERS_directed_final.parquet` | Directed road network ready for criticality analysis |
| `intermediate_results/giant_component_dropped_roads.parquet` | Edges excluded by the giant component filter (for inspection) |
| `figures/{traffic_type}_aadt_map.png` | Individual AADT map per traffic type (6 files) |
| `figures/AADT_categories_combined.png` | 6-panel combined AADT category map |

---

## Workflow

### 1. Load and deduplicate raw network
Read `DeoniceRSDP-Jul2025_corrected_topology.parquet`, retain only the relevant attribute columns, and drop exact duplicate features (identical attributes **and** geometry).

### 2. Snap endpoints iteratively
Snap road endpoints to nearby endpoints or road segments using a **2 m snap tolerance** and **30 m search buffer**. Runs until no further snaps occur (max 20 iterations). Resolves small geometric gaps that would otherwise break network connectivity.

### 3. Build initial topology
- Add start/end node points to each edge
- Split edges at intersection nodes (where one road crosses mid-segment of another)
- Re-add endpoints, assign integer IDs, derive `from_id` / `to_id` topology

### 4. Load and merge AADT data
Load `PGDS_2024.shp` and merge onto the network in two passes:
1. **Exact match** on `oznaka_deo` segment identifier
2. **Spatial join** for unmatched segments: retain matches with **≥ 50% geometric overlap**, taking the `max` AADT value where multiple AADT segments overlap a single road

### 5. Fill missing AADT
Roads still missing AADT after the merge are filled in two passes:
- **Pass 1** — if both endpoints touch roads with known AADT, assign the average of those touching values
- **Pass 2** — assign the median AADT for the road's `kategorija` class, then cap at the maximum AADT of any touching road with a known value

### 6. Filter to Serbia
Exclude roads whose geometry intersects the Kosovo boundary polygon (Natural Earth `SOV_A3 == 'KOS'`).

### 7. Visualise AADT
Generate and save choropleth maps for six traffic types (passenger cars, buses, light trucks, medium trucks, heavy trucks, articulated vehicles), individually and as a combined 6-panel figure.

### 8. Rebuild topology on Serbia network
Re-run endpoint detection, ID assignment, and topology derivation on the Kosovo-filtered network (`AADT_Serbia`).

### 9. Create directed graph
- Identify bidirectional roads: `smer_gdf1` not in `['L', 'D']` (values `'O'`, `None`, `'M'`)
- **Halve AADT** on bidirectional roads — source data records the two-direction combined total
- Append a **reversed geometry copy** of each bidirectional edge, yielding a fully directed edge set

### 10. Compute travel time attributes

| Column | Formula | Notes |
|--------|---------|-------|
| `road_length` | `shapely.length(geometry) / 1000` | km, from projected geometry |
| `speed` | lookup by `kategorija` | IM / IA / IB = 100 km/h; IIA / IIB = 80 km/h; default = 80 km/h |
| `fft` | `road_length / speed` | free-flow travel time in hours |

Any NaN / inf `fft` values are flagged with a detailed diagnostic report. **No automatic correction is applied** — inspect flagged rows and fix upstream if needed.

### 11. Extract strongly-connected giant component
Load into igraph as a directed graph and retain the strongly-connected giant component. A diagnostic check reports:
- Number and percentage of dropped edges
- Road length retained vs. total (km)
- Saves a map and raises an assertion error if the giant component is **< 95%** of total network length

### 12. Save
Save the giant component edge table to `intermediate_results/PERS_directed_final.parquet`.

In [ ]:
# Standard library
import os
import re
import sys
import warnings
from pathlib import Path

# Third-party - Data and scientific computing
import contextily as cx
import geopandas as gpd
import igraph as ig
import numpy as np
import pandas as pd
from pyproj import Geod
from tqdm import tqdm

# Shapely-specific imports for spatial analysis
import shapely
from shapely import STRtree
from shapely.geometry import LineString, MultiLineString, Point
from shapely.ops import nearest_points, snap

# Matplotlib-specific imports for figures
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import FuncFormatter, MultipleLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Suppress warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=RuntimeWarning)

In [ ]:
BASE_DIR = Path.cwd().parent   # if notebook is in /notebooks

sys.path.append(str(BASE_DIR))
from src.simplify import *

In [ ]:
data_path = BASE_DIR / "input_files"
figure_path = BASE_DIR / "figures"
intermediate_path = BASE_DIR / 'intermediate_results'

intermediate_path.mkdir(parents=True, exist_ok=True)

roads_path = data_path / "DeoniceRSDP-Jul2025_corrected_topology.parquet"
pers_network = gpd.read_parquet(roads_path)

In [ ]:
attributes = ['objectid','oznaka_deo','smer_gdf1','kategorija','oznaka_put','oznaka_poc','naziv_poce', 'oznaka_zav', 'naziv_zavr', 'duzina_deo',
       'pocetna_st', 'zavrsna_st','stanje','geometry']                                                

pers_network = pers_network[attributes]

# Check and remove exact duplicates (including geometry)
total_rows = len(pers_network)
duplicate_mask = pers_network.duplicated(keep=False)
unique_duplicated = pers_network[duplicate_mask].drop_duplicates().shape[0]
total_duplicates = duplicate_mask.sum()

pers_network = pers_network.drop_duplicates()
pers_network = pers_network.reset_index(drop=True)

print(f"  Total rows before deduplication         : {total_rows}")
print(f"  Unique rows that have a duplicate       : {unique_duplicated}")
print(f"  Total rows involved in duplication      : {total_duplicates}")
print(f"  Total rows after deduplication          : {len(pers_network)}")

In [ ]:
def snap_network_iteratively(gdf, tolerance=2, search_buffer=30):
    """
    Iteratively snap road endpoints to nearby endpoints OR nearby road segments.
    Uses spatial index for efficiency.
    """
    
    def get_endpoints(geom):
        if geom is None or geom.is_empty:
            return None, None
        
        if isinstance(geom, MultiLineString):
            lines = list(geom.geoms)
            if len(lines) == 0:
                return None, None
            first_line = lines[0]
            last_line = lines[-1]
            start_coords = list(first_line.coords)[0]
            end_coords = list(last_line.coords)[-1]
            return Point(start_coords), Point(end_coords)
        
        elif isinstance(geom, LineString):
            coords = list(geom.coords)
            return Point(coords[0]), Point(coords[-1])
        
        else:
            return None, None
    
    gdf = gdf.copy()
    total_snaps = 0
    indices = list(gdf.index)
    
    # Track pairs that have already been snapped
    already_snapped_pairs = set()
    
    iteration = 0
    while True:
        iteration += 1
        snaps_this_round = 0
        snapped_this_round = set()
        
        # Build spatial index fresh each iteration
        geometries = gdf.geometry.tolist()
        tree = STRtree(geometries)
        idx_to_pos = {idx: i for i, idx in enumerate(indices)}
        pos_to_idx = {i: idx for i, idx in enumerate(indices)}
        
        for idx1 in tqdm(indices, desc=f"Iteration {iteration}"):
            if idx1 in snapped_this_round:
                continue
            
            geom1 = gdf.loc[idx1, 'geometry']
            if geom1 is None or geom1.is_empty:
                continue
                
            start1, end1 = get_endpoints(geom1)
            if start1 is None:
                continue
            
            # Find candidate roads within buffer
            buffer_geom = geom1.buffer(search_buffer)
            candidate_positions = tree.query(buffer_geom)
            candidate_indices = [pos_to_idx[pos] for pos in candidate_positions]
            
            for pt1, pos1 in [(start1, 'start'), (end1, 'end')]:
                if idx1 in snapped_this_round:
                    break
                
                for idx2 in candidate_indices:
                    if idx1 == idx2:
                        continue
                    
                    # Skip if this pair was already snapped
                    pair = tuple(sorted([idx1, idx2]))
                    if pair in already_snapped_pairs:
                        continue
                    
                    geom2 = gdf.loc[idx2, 'geometry']
                    if geom2 is None or geom2.is_empty:
                        continue
                    
                    dist = pt1.distance(geom2)
                    
                    if 0 < dist <= tolerance:
                        gdf.loc[idx1, 'geometry'] = snap(
                            gdf.loc[idx1, 'geometry'], 
                            geom2, 
                            tolerance
                        )
                        snapped_this_round.add(idx1)
                        already_snapped_pairs.add(pair)
                        snaps_this_round += 1
                        
                        start1, end1 = get_endpoints(gdf.loc[idx1, 'geometry'])
                        break
        
        print(f"Iteration {iteration} complete: {snaps_this_round} snaps")
        total_snaps += snaps_this_round
        
        if snaps_this_round == 0:
            break
        if iteration > 20:
            print("Max iterations reached")
            break
    
    print(f"\nTotal snaps made: {total_snaps}")
    return gdf

In [ ]:
# Run it
pers_network = snap_network_iteratively(pers_network, tolerance=2, search_buffer=30)

In [ ]:
# Create a Network object from the input DataFrame
net = Network(edges=pers_network)

In [ ]:
net = add_endpoints(net)
net = split_edges_at_nodes(net,attributes=['objectid','oznaka_deo','smer_gdf1','kategorija',
                                           'oznaka_put','oznaka_poc','naziv_poce', 'oznaka_zav', 
                                           'naziv_zavr', 'duzina_deo','pocetna_st', 'zavrsna_st','stanje'])
net = add_endpoints(net)
net = add_ids(net)
net = add_topology(net)    

In [ ]:
pers_network = net.edges.set_crs(pers_network.crs)

In [ ]:
AADT_path = data_path / "PGDS_2024.shp"
aadt_network = gpd.read_file(AADT_path)

In [ ]:
aadt_cols = ['PA', 'BUS', 'LT', 'ST', 'TT', 'AV', 'Ukupno']
aadt_network.dropna(subset=aadt_cols, inplace=True)

In [ ]:
aadt_network

In [ ]:
traffic_types = ['passenger_cars', 'buses', 'light_trucks', 'medium_trucks',
                 'heavy_trucks', 'articulated_vehicles', 'total_aadt']

In [ ]:
aadt_cols = ['PA', 'BUS', 'LT', 'ST', 'TT', 'AV', 'Ukupno']
aadt_network.dropna(subset=aadt_cols, inplace=True)

# CRS check — reproject pers_network if it doesn't match aadt_network
if pers_network.crs is None:
    print(f"⚠ pers_network has no CRS — assuming same as aadt_network ({aadt_network.crs})")
    pers_network = pers_network.set_crs(aadt_network.crs)
elif pers_network.crs != aadt_network.crs:
    print(f"⚠ CRS mismatch — reprojecting pers_network from {pers_network.crs} to {aadt_network.crs}")
    pers_network = pers_network.to_crs(aadt_network.crs)

# First merge on oznaka_deo
first_merger = pers_network.merge(
    aadt_network[aadt_cols+['oznaka_deo']], 
    how='left', 
    left_on='oznaka_deo', 
    right_on='oznaka_deo'
)

# Deduplicate: for each original network row, keep max AADT values
agg_dict = {col: 'first' for col in pers_network.columns if col in first_merger.columns}
agg_dict.update({col: 'max' for col in aadt_cols})
first_merger = first_merger.groupby(level=0).agg(agg_dict)
first_merger = gpd.GeoDataFrame(first_merger, geometry='geometry')

# Spatial join for unmatched rows
overlap = first_merger.loc[first_merger.PA.isna()][pers_network.columns].sjoin(
    aadt_network[aadt_cols+['oznaka_deo','geometry']], 
    how='left', 
    predicate='intersects'
)

# Restore oznaka_deo from the left side after the sjoin suffix collision
# (keeps the original road identifier on spatially-matched segments)
if 'oznaka_deo_left' in overlap.columns:
    overlap = overlap.rename(columns={'oznaka_deo_left': 'oznaka_deo'})

# Get the AADT geometries for the matched index_right values
overlap_with_aadt_geom = overlap.dropna(subset=['index_right']).copy()
overlap_with_aadt_geom['aadt_geometry'] = aadt_network.loc[overlap_with_aadt_geom['index_right'].astype(int), 'geometry'].values

# Calculate intersection and overlap ratio
overlap_with_aadt_geom['intersection_geom'] = overlap_with_aadt_geom.apply(
    lambda row: row['geometry'].intersection(row['aadt_geometry']), axis=1
)
overlap_with_aadt_geom['overlap_ratio'] = (
    overlap_with_aadt_geom['intersection_geom'].length / overlap_with_aadt_geom['geometry'].length
)

# Keep only rows with >= 50% overlap
overlap_filtered = overlap_with_aadt_geom[overlap_with_aadt_geom['overlap_ratio'] >= 0.5].copy()
overlap_filtered = overlap_filtered.drop(columns=['aadt_geometry', 'intersection_geom', 'overlap_ratio'])

first_cols = ['oznaka_deo', 'smer_gdf1', 'kategorija', 'oznaka_put', 'oznaka_poc', 
              'naziv_poce', 'oznaka_zav', 'naziv_zavr', 'duzina_deo', 'pocetna_st', 
              'geometry']

agg_dict = {col: 'first' for col in first_cols if col in overlap_filtered.columns}
agg_dict.update({col: 'max' for col in aadt_cols})

result = overlap_filtered.dropna(subset=aadt_cols).groupby(level=0).agg(agg_dict)

# Continue with concatenation...
AADT_connected = pd.concat([first_merger.loc[first_merger.dropna(subset=aadt_cols).index], result])
AADT_connected = gpd.GeoDataFrame(pd.concat([AADT_connected,pers_network.loc[~pers_network.index.isin(AADT_connected.index)]]))

AADT_connected = AADT_connected.rename(columns = {
    'PA' : 'passenger_cars', 
    'BUS': 'buses', 
    'LT': 'light_trucks', 
    'ST': 'medium_trucks',  
    'TT': 'heavy_trucks', 
    'AV': 'articulated_vehicles', 
    'Ukupno': 'total_aadt'
}
                     )

AADT_connected[traffic_types] = AADT_connected[traffic_types].astype(np.float64)

# Ensure it's a GeoDataFrame with proper CRS
AADT_connected = gpd.GeoDataFrame(AADT_connected, geometry='geometry')

traffic_cols = ['passenger_cars', 'buses', 'light_trucks', 'medium_trucks', 
                'heavy_trucks', 'articulated_vehicles', 'total_aadt']

In [ ]:
def get_endpoints(geom):
    """Get start and end points of a linestring or multilinestring"""
    if geom is None or geom.is_empty:
        return None, None
    
    # Handle MultiLineString
    if isinstance(geom, MultiLineString):
        # Get all component linestrings
        lines = list(geom.geoms)
        if len(lines) == 0:
            return None, None
        # Get start of first line and end of last line
        first_line = lines[0]
        last_line = lines[-1]
        start_coords = list(first_line.coords)[0]
        end_coords = list(last_line.coords)[-1]
        return Point(start_coords), Point(end_coords)
    
    # Handle regular LineString
    elif isinstance(geom, LineString):
        coords = list(geom.coords)
        return Point(coords[0]), Point(coords[-1])
    
    else:
        return None, None

def find_touching_roads_with_aadt(idx, gdf, buffer_dist=1):
    """Find roads that touch the endpoints of a given road and have AADT values"""
    row = gdf.loc[idx]
    start_pt, end_pt = get_endpoints(row.geometry)
    
    if start_pt is None:
        return []
    
    touching_roads = []
    for other_idx, other_row in gdf.iterrows():
        if other_idx == idx:
            continue
        if pd.isna(other_row['total_aadt']):
            continue
            
        # Check if endpoints touch the other road
        if other_row.geometry is not None and not other_row.geometry.is_empty:
            if start_pt.buffer(buffer_dist).intersects(other_row.geometry):
                touching_roads.append(('start', other_idx, other_row))
            if end_pt.buffer(buffer_dist).intersects(other_row.geometry):
                touching_roads.append(('end', other_idx, other_row))
    
    return touching_roads

# ============================================
# PASS 1: Fill from both endpoints touching roads with AADT
# ============================================
print("Pass 1: Filling from roads touching both endpoints...")

missing_aadt = AADT_connected[AADT_connected['total_aadt'].isna()].index.tolist()
filled_count_pass1 = 0

for idx in tqdm(missing_aadt,total=len(missing_aadt)):
    touching = find_touching_roads_with_aadt(idx, AADT_connected)
    
    # Check if we have at least one touch at start and one at end
    start_touches = [t for t in touching if t[0] == 'start']
    end_touches = [t for t in touching if t[0] == 'end']
    
    if len(start_touches) > 0 and len(end_touches) > 0:
        # Get AADT values from touching roads
        start_values = {col: np.mean([t[2][col] for t in start_touches]) for col in traffic_cols}
        end_values = {col: np.mean([t[2][col] for t in end_touches]) for col in traffic_cols}
        
        # Take average of start and end
        for col in traffic_cols:
            AADT_connected.loc[idx, col] = (start_values[col] + end_values[col]) / 2
        
        filled_count_pass1 += 1

print(f"Pass 1 filled {filled_count_pass1} roads")

# ============================================
# PASS 2: Fill with median by kategorija, then cap by touching roads
# ============================================
print("Pass 2: Filling with kategorija median...")

# Calculate median values per kategorija
kategoria_medians = AADT_connected.groupby('kategorija')[traffic_cols].median()

missing_aadt = AADT_connected[AADT_connected['total_aadt'].isna()].index.tolist()
filled_count_pass2 = 0

for idx in tqdm(missing_aadt,total=len(missing_aadt)):
    row = AADT_connected.loc[idx]
    kategorija = row['kategorija']
    
    # Skip if no kategorija
    if pd.isna(kategorija) or kategorija not in kategoria_medians.index:
        continue
    
    # Fill with median values
    median_values = kategoria_medians.loc[kategorija]
    for col in traffic_cols:
        AADT_connected.loc[idx, col] = median_values[col]
    
    # Now check touching roads and cap if our value is higher
    touching = find_touching_roads_with_aadt(idx, AADT_connected)
    
    if len(touching) > 0:
        # Get max AADT from any touching road
        max_touching_values = {col: max([t[2][col] for t in touching]) for col in traffic_cols}
        
        # Cap our values if they exceed touching roads
        for col in traffic_cols:
            if AADT_connected.loc[idx, col] > max_touching_values[col]:
                AADT_connected.loc[idx, col] = max_touching_values[col]
    
    filled_count_pass2 += 1

print(f"Pass 2 filled {filled_count_pass2} roads")

# Summary
remaining_missing = AADT_connected['total_aadt'].isna().sum()
print(f"\nRemaining roads without AADT: {remaining_missing}")

In [ ]:
# Load country outline
world_path = data_path / "ne_10m_admin_0_countries.shp"
world = gpd.read_file(world_path)
country = world.loc[world.SOV_A3 == 'KOS']
country = country.to_crs(AADT_connected.crs)

# Dissolve in case there are multiple polygons
kosovo_geom = country.union_all()  # or country.unary_union for older geopandas

# Filter roads that are within Serbia
AADT_Serbia = AADT_connected[~AADT_connected.geometry.intersects(kosovo_geom)].copy()

### PREPARE SEVERAL MAPS TO CHECK THE RESULTS (AND FOR THE REPORT)

In [ ]:
gdf_aadt = AADT_Serbia.copy()

traffic_types = ['passenger_cars', 'buses', 'light_trucks', 'medium_trucks',
                 'heavy_trucks', 'articulated_vehicles', 'total_aadt']

colors = ['#005f73','#9b2226','#a53860','#283618','#2a9d8f','#582f0e','#001219']

legend_titles = {
    'passenger_cars': 'Passenger Cars AADT (vehicles/day)',
    'buses': 'Buses AADT (vehicles/day)', 
    'light_trucks': 'Light Trucks AADT (vehicles/day)',
    'medium_trucks': 'Medium Trucks AADT (vehicles/day)',
    'heavy_trucks': 'Heavy Trucks AADT (vehicles/day)',
    'articulated_vehicles': 'Articulated Vehicles AADT (vehicles/day)',
    'total_aadt': 'Total AADT (vehicles/day)'
}

breaks_labels = {
    'passenger_cars': ([0, 5000, 10000, 20000, 30000, float('inf')], 
                      ['< 5,000', '5,000-10,000', '10,000-20,000', '20,000-30,000', '> 30,000']),
    'buses': ([0, 50, 100, 200, 400, float('inf')], 
              ['< 50', '50-100', '100-200', '200-400', '> 400']),
    'light_trucks': ([0, 100, 200, 400, 600, float('inf')], 
                    ['< 100', '100-200', '200-400', '400-600', '> 600']),
    'medium_trucks': ([0, 100, 200, 400, 600, float('inf')], 
                     ['< 100', '100-200', '200-400', '400-600', '> 600']),
    'heavy_trucks': ([0, 50, 100, 200, 300, float('inf')], 
                    ['< 50', '50-100', '100-200', '200-300', '> 300']),
    'articulated_vehicles': ([0, 1000, 2000, 4000, 6000, float('inf')], 
                           ['< 1,000', '1,000-2,000', '2,000-4,000', '4,000-6,000', '> 6,000']),
    'total_aadt': ([0, 5000, 10000, 20000, 40000, float('inf')], 
                  ['< 5,000', '5,000-10,000', '10,000-20,000', '20,000-40,000', '> 40,000'])
}

width_mappings = {
    'passenger_cars': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 5,000', '5,000-10,000', '10,000-20,000', '20,000-30,000', '> 30,000'])},
    'buses': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 50', '50-100', '100-200', '200-400', '> 400'])},
    'light_trucks': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 100', '100-200', '200-400', '400-600', '> 600'])},
    'medium_trucks': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 100', '100-200', '200-400', '400-600', '> 600'])},
    'heavy_trucks': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 50', '50-100', '100-200', '200-300', '> 300'])},
    'articulated_vehicles': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 1,000', '1,000-2,000', '2,000-4,000', '4,000-6,000', '> 6,000'])},
    'total_aadt': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 5,000', '5,000-10,000', '10,000-20,000', '20,000-40,000', '> 40,000'])}
}

for i, traffic_type in enumerate(traffic_types[:6]):
    breaks, labels = breaks_labels[traffic_type]
    
    # Create categories for this traffic type
    gdf_aadt[f'{traffic_type}_category'] = pd.cut(gdf_aadt[traffic_type], 
                                                  bins=breaks, labels=labels, include_lowest=True)
    
    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    for category in labels:
        subset = gdf_aadt[gdf_aadt[f'{traffic_type}_category'] == category]
        if len(subset) > 0:
            width = width_mappings[traffic_type][category]
            subset.plot(ax=ax, color=colors[i], alpha=0.7,
                       linewidth=width, label=category)
    
    cx.add_basemap(ax=ax, crs=gdf_aadt.crs.to_string(),
                    source=cx.providers.OpenStreetMap.Mapnik, 
                    alpha=0.3, attribution=False)
    ax.legend(title=legend_titles[traffic_type], loc='upper right')
    ax.axis('off')
    plt.savefig(figure_path / f'{traffic_type}_aadt_map.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
gdf_aadt = AADT_Serbia.copy()

# Exclude 'total_aadt' - only 6 categories
traffic_types = ['passenger_cars', 'buses', 'light_trucks', 'medium_trucks',
                 'heavy_trucks', 'articulated_vehicles']
colors = ['#005f73', '#9b2226', '#a53860', '#283618', '#2a9d8f', '#582f0e']
letters = ['A', 'B', 'C', 'D', 'E', 'F']

legend_titles = {
    'passenger_cars': 'Passenger Cars\n(vehicles/day)',
    'buses': 'Buses\n(vehicles/day)', 
    'light_trucks': 'Light Trucks\n(vehicles/day)',
    'medium_trucks': 'Medium Trucks\n(vehicles/day)',
    'heavy_trucks': 'Heavy Trucks\n(vehicles/day)',
    'articulated_vehicles': 'Articulated Vehicles\n(vehicles/day)'
}

breaks_labels = {
    'passenger_cars': ([0, 5000, 10000, 20000, 30000, float('inf')], 
                      ['< 5,000', '5,000-10,000', '10,000-20,000', '20,000-30,000', '> 30,000']),
    'buses': ([0, 50, 100, 200, 400, float('inf')], 
              ['< 50', '50-100', '100-200', '200-400', '> 400']),
    'light_trucks': ([0, 100, 200, 400, 600, float('inf')], 
                    ['< 100', '100-200', '200-400', '400-600', '> 600']),
    'medium_trucks': ([0, 100, 200, 400, 600, float('inf')], 
                     ['< 100', '100-200', '200-400', '400-600', '> 600']),
    'heavy_trucks': ([0, 50, 100, 200, 300, float('inf')], 
                    ['< 50', '50-100', '100-200', '200-300', '> 300']),
    'articulated_vehicles': ([0, 1000, 2000, 4000, 6000, float('inf')], 
                           ['< 1,000', '1,000-2,000', '2,000-4,000', '4,000-6,000', '> 6,000'])
}

width_mappings = {
    'passenger_cars': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 5,000', '5,000-10,000', '10,000-20,000', '20,000-30,000', '> 30,000'])},
    'buses': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 50', '50-100', '100-200', '200-400', '> 400'])},
    'light_trucks': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 100', '100-200', '200-400', '400-600', '> 600'])},
    'medium_trucks': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 100', '100-200', '200-400', '400-600', '> 600'])},
    'heavy_trucks': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 50', '50-100', '100-200', '200-300', '> 300'])},
    'articulated_vehicles': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 1,000', '1,000-2,000', '2,000-4,000', '4,000-6,000', '> 6,000'])}
}

# Create figure with 3 rows x 2 columns
fig, axes = plt.subplots(3, 2, figsize=(12, 24), facecolor='white')
axes = axes.flatten()  # Flatten to easily iterate

for i, traffic_type in enumerate(traffic_types):
    ax = axes[i]
    breaks, labels = breaks_labels[traffic_type]
    
    # Create categories for this traffic type
    gdf_aadt[f'{traffic_type}_category'] = pd.cut(
        gdf_aadt[traffic_type], 
        bins=breaks, labels=labels, include_lowest=True
    )
    
    # Plot each category
    for category in labels:
        subset = gdf_aadt[gdf_aadt[f'{traffic_type}_category'] == category]
        if len(subset) > 0:
            width = width_mappings[traffic_type][category]
            subset.plot(ax=ax, color=colors[i], alpha=0.7, linewidth=width)
    
    # Add basemap
    cx.add_basemap(ax=ax, crs=gdf_aadt.crs.to_string(),
                   source=cx.providers.OpenStreetMap.Mapnik, 
                   alpha=0.3, attribution=False)
    
    # Create legend with line widths
    legend_elements = [
        Line2D([0], [0], color=colors[i], lw=width_mappings[traffic_type][cat], 
               label=cat, alpha=0.7)
        for cat in labels
    ]
    ax.legend(handles=legend_elements, title=legend_titles[traffic_type], 
              loc='upper right', fontsize=13, title_fontsize=15,
              frameon=True, fancybox=True, shadow=True,
              framealpha=0.9, facecolor='white', edgecolor='#cccccc')
    
    ax.axis('off')
    
    # Add letter label
    ax.text(0.05, 0.95, letters[i], transform=ax.transAxes, fontsize=20, 
            fontweight='bold', verticalalignment='top',
            bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(figure_path /'AADT_categories_combined.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
gdf_aadt = aadt_network.copy()
gdf_aadt = gdf_aadt.rename(columns = {
    'PA' : 'passenger_cars', 
    'BUS': 'buses', 
    'LT': 'light_trucks', 
    'ST': 'medium_trucks',  
    'TT': 'heavy_trucks', 
    'AV': 'articulated_vehicles', 
    'Ukupno': 'total_aadt'
}
                     )

gdf_aadt[traffic_types] = gdf_aadt[traffic_types].astype(np.float64)


traffic_types = ['passenger_cars', 'buses', 'light_trucks', 'medium_trucks',
                 'heavy_trucks', 'articulated_vehicles', 'total_aadt']

colors = ['#005f73','#9b2226','#a53860','#283618','#2a9d8f','#582f0e','#001219']

legend_titles = {
    'passenger_cars': 'Passenger Cars AADT (vehicles/day)',
    'buses': 'Buses AADT (vehicles/day)', 
    'light_trucks': 'Light Trucks AADT (vehicles/day)',
    'medium_trucks': 'Medium Trucks AADT (vehicles/day)',
    'heavy_trucks': 'Heavy Trucks AADT (vehicles/day)',
    'articulated_vehicles': 'Articulated Vehicles AADT (vehicles/day)',
    'total_aadt': 'Total AADT (vehicles/day)'
}

breaks_labels = {
    'passenger_cars': ([0, 5000, 10000, 20000, 30000, float('inf')], 
                      ['< 5,000', '5,000-10,000', '10,000-20,000', '20,000-30,000', '> 30,000']),
    'buses': ([0, 50, 100, 200, 400, float('inf')], 
              ['< 50', '50-100', '100-200', '200-400', '> 400']),
    'light_trucks': ([0, 100, 200, 400, 600, float('inf')], 
                    ['< 100', '100-200', '200-400', '400-600', '> 600']),
    'medium_trucks': ([0, 100, 200, 400, 600, float('inf')], 
                     ['< 100', '100-200', '200-400', '400-600', '> 600']),
    'heavy_trucks': ([0, 50, 100, 200, 300, float('inf')], 
                    ['< 50', '50-100', '100-200', '200-300', '> 300']),
    'articulated_vehicles': ([0, 1000, 2000, 4000, 6000, float('inf')], 
                           ['< 1,000', '1,000-2,000', '2,000-4,000', '4,000-6,000', '> 6,000']),
    'total_aadt': ([0, 5000, 10000, 20000, 40000, float('inf')], 
                  ['< 5,000', '5,000-10,000', '10,000-20,000', '20,000-40,000', '> 40,000'])
}

width_mappings = {
    'passenger_cars': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 5,000', '5,000-10,000', '10,000-20,000', '20,000-30,000', '> 30,000'])},
    'buses': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 50', '50-100', '100-200', '200-400', '> 400'])},
    'light_trucks': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 100', '100-200', '200-400', '400-600', '> 600'])},
    'medium_trucks': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 100', '100-200', '200-400', '400-600', '> 600'])},
    'heavy_trucks': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 50', '50-100', '100-200', '200-300', '> 300'])},
    'articulated_vehicles': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 1,000', '1,000-2,000', '2,000-4,000', '4,000-6,000', '> 6,000'])},
    'total_aadt': {cat: 0.5 + i * 0.75 for i, cat in enumerate(['< 5,000', '5,000-10,000', '10,000-20,000', '20,000-40,000', '> 40,000'])}
}

for i, traffic_type in enumerate(traffic_types[6:]):
    breaks, labels = breaks_labels[traffic_type]
    
    # Create categories for this traffic type
    gdf_aadt[f'{traffic_type}_category'] = pd.cut(gdf_aadt[traffic_type].astype(np.float64), bins=breaks, labels=labels, include_lowest=True)
    
    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    
    for category in labels:
        subset = gdf_aadt[gdf_aadt[f'{traffic_type}_category'] == category]
        if len(subset) > 0:
            width = width_mappings[traffic_type][category]
            subset.plot(ax=ax, color=colors[i], alpha=0.7,
                       linewidth=width, label=category)
    
    cx.add_basemap(ax=ax, crs=gdf_aadt.crs.to_string(),
                    source=cx.providers.OpenStreetMap.Mapnik, 
                    alpha=0.3, attribution=False)
    ax.legend(title=legend_titles[traffic_type], loc='upper right')
    ax.axis('off')
    plt.savefig(figure_path / f'{traffic_type}_aadt_map_og.png', dpi=300, bbox_inches='tight')
    plt.show()

## Prepare creation of fully connected network with correct attributes 

In [ ]:
attributes = ['oznaka_deo', 'smer_gdf1', 'kategorija', 'oznaka_put', 'oznaka_poc',
       'naziv_poce', 'oznaka_zav', 'naziv_zavr', 'duzina_deo', 'pocetna_st',
       'zavrsna_st', 'stanje','passenger_cars', 'buses','light_trucks', 'medium_trucks', 'heavy_trucks', 'articulated_vehicles',
       'total_aadt']

In [ ]:
net = Network(edges=AADT_Serbia)
net = add_endpoints(net)
net = add_ids(net)
net = add_topology(net)    
base_network = net.edges.set_crs(AADT_Serbia.crs)

In [ ]:
# Step 1: Filter for roads that are not oneway
non_oneway_mask = ~base_network['smer_gdf1'].isin(['L', 'D'])

# Diagnostic: print all smer_gdf1 values present on non-oneway roads
smer_counts = base_network.loc[non_oneway_mask, 'smer_gdf1'].value_counts(dropna=False)
print(f"Non-oneway smer_gdf1 values (total {non_oneway_mask.sum()} roads):")
for val, count in smer_counts.items():
    print(f"  {repr(val)}: {count}")

# Halve AADT on bidirectional roads — original values are sum of both directions
base_network.loc[non_oneway_mask, traffic_types] = base_network.loc[non_oneway_mask, traffic_types] / 2
non_oneway_roads = base_network[non_oneway_mask]

# Step 2: Create reverse edges
def reverse_road(row):
    reversed_geometry = shapely.LineString(row['geometry'].coords[::-1])
    new_row = row.copy()
    new_row['from_id'], new_row['to_id'] = row['to_id'], row['from_id']
    new_row['geometry'] = reversed_geometry
    return new_row

# Step 3: Apply the reverse function to each row
reversed_edges = non_oneway_roads.apply(reverse_road, axis=1)

# Step 4: Append reversed edges back to the original GeoDataFrame
base_network = gpd.GeoDataFrame(pd.concat([base_network, reversed_edges])).reset_index(drop=True)
base_network['id'] = base_network.index

In [ ]:
speed_d = {"IM": 100,
        "IA": 100,
        "IB": 100,
        "IIA": 80,
        "IIB": 80}

def fill_speed(x):
    try:
        return speed_d[x.kategorija]
    except Exception:
        return 80

geod = Geod(ellps="WGS84")
base_network['road_length'] = base_network.geometry.apply(lambda line_string: shapely.length(line_string)/1e3)
base_network['speed'] = base_network.apply(lambda x: fill_speed(x),axis=1)
base_network['fft'] = base_network.apply(lambda x : ((x.road_length)/x.speed),axis=1)

bad_mask = ~np.isfinite(base_network['fft'])
if bad_mask.any():
    raw_geom_m = base_network.loc[bad_mask].geometry.length
    detail_cols = [c for c in ['oznaka_deo', 'kategorija', 'stanje', 'duzina_deo'] if c in base_network.columns]
    report = base_network.loc[bad_mask, detail_cols].copy()
    report['geom_length_m'] = raw_geom_m.values
    report['road_length_km'] = base_network.loc[bad_mask, 'road_length'].values
    report['fft'] = base_network.loc[bad_mask, 'fft'].values
    print(f"WARNING: {bad_mask.sum()} edge(s) with NaN/inf fft — inspect before proceeding:")
    print(report.to_string())
    print("  ^^^ Compare geom_length_m against duzina_deo — large duzina_deo with NaN geom means corrupted geometry.")
else:
    print("fft check: all edges have valid fft values.")

In [ ]:
## Load the baseline into igraph:
edges = base_network.reindex(['from_id','to_id'] + [x for x in list(base_network.columns) if x not in ['from_id','to_id']],axis=1)
graph = ig.Graph.TupleList(edges.itertuples(index=False), edge_attrs=list(edges.columns)[2:],directed=True)
graph = graph.connected_components().giant()
edges = edges[edges['id'].isin(graph.es['id'])]

In [ ]:
dropped_mask = ~base_network["id"].isin(edges["id"])
n_dropped = int(dropped_mask.sum())

if n_dropped == 0:
    print("Giant component check: no roads dropped, full network is connected.")
else:
    total_length = base_network["road_length"].sum()
    giant_length = base_network.loc[~dropped_mask, "road_length"].sum()
    dropped_length = base_network.loc[dropped_mask, "road_length"].sum()
    length_fraction = giant_length / total_length

    print(
        f"Giant component check: {n_dropped} of {len(base_network)} roads "
        f"({100 * n_dropped / len(base_network):.2f}%) dropped."
    )
    print(
        f"  By number of roads : kept {len(base_network) - n_dropped}/{len(base_network)} "
        f"({100 * (len(base_network) - n_dropped) / len(base_network):.2f}%)"
    )
    print(
        f"  By road length     : kept {giant_length:.2f}/{total_length:.2f} km "
        f"({100 * length_fraction:.2f}%), dropped {dropped_length:.2f} km"
    )

    if length_fraction < 0.95:
        fig, ax = plt.subplots(1, 1, figsize=(16, 8))
        base_network.loc[~dropped_mask].plot(ax=ax, color="grey", linewidth=0.5, label="Included")
        base_network.loc[dropped_mask].plot(ax=ax, color="red", linewidth=1.5, label="Dropped")
        ax.legend(title="Giant component check", loc="upper right")
        ax.axis("off")
        plt.savefig(figure_path / "giant_component_dropped_roads.png", dpi=300, bbox_inches="tight")
        plt.show()

        assert length_fraction >= 0.95, (
            f"Giant component represents only {100 * length_fraction:.2f}% of the original "
            f"network length — below the required 95% threshold."
        )


### And save the network for further analysis

In [ ]:
edges.reset_index(drop=True).set_crs(AADT_Serbia.crs).to_parquet(intermediate_path / 'PERS_directed_final.parquet')